# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — Baseline (Qwen2.5-14B-Instruct, no FT) on the Stage 3 Test Split

Row-aligned counterpart to `stage_1_inference_on_FT_natural_canonical.ipynb`.
Same Stage 3 test split (same row IDs, same `row_seed`, same letter permutations),
different model weights — so any per-row p_true delta is attributable to
fine-tuning rather than sampling variability.

## Why this notebook exists

The original baseline (`Qwen3-4B-Instruct-2507`) used `build_eval_set` to resample
rows from the full ACS extract independently per target. That sampling is **not**
row-aligned with the Stage 3 test split, so:

- **Group-level comparisons** (per-class κ, F1, ΔH_c, mean p_true) between the old
  baseline and the new FT run are still valid — both estimate the same
  population-level quantities.
- **Within-person comparisons** (paired bootstrap on Δp_true, McNemar on correct↔incorrect
  flips, per-row improvement decomposition) are **not** valid — there's no row
  correspondence.

This notebook produces a baseline run on the Stage 3 test split itself, so paired
tests become well-defined.

## Key choices

| Field | Value | Note |
|---|---|---|
| Model | `Qwen/Qwen2.5-14B-Instruct` | Same architecture/tokenizer as the FT, so any delta is the FT effect |
| Adapter | none | This is the control |
| Eval set | Stage 3 `test/{TEST_KIND}/<TARGET>.parquet` | Identical to the FT notebook |
| `row_seed` | inherited from Stage 3 parquet | Letter permutations match the FT run row-by-row |
| `RANDOM_SEED` | 4242 | Distinct from FT (4343) for book-keeping only |

## TEST_KIND — run this notebook twice

Set `TEST_KIND = "balanced"` for the first pass, then `"raw"` for the second.
The output directory carries `TEST_KIND` in its tag so both runs sit side-by-side
and each is row-aligned with the corresponding FT pass.


## 0. Setup

In [ ]:
# Optional install (uncomment in Colab on a fresh runtime)
!pip install -q vllm pandas numpy tqdm pyarrow transformers

In [ ]:

!pip install -q -U torchao

In [ ]:
import os

# --- vLLM environment knobs -------------------------------------------------
#
# Must be set BEFORE `import vllm` anywhere in the kernel. If you've already
# imported vllm in this Colab session, do Runtime → Restart session and
# re-run from the top.
#
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"]   = "spawn"

import gc
import io
import json
import math
import string
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

# Patch sys.stdout.fileno to avoid io.UnsupportedOperation in Jupyter/Colab.
try:
    sys.stdout.fileno()
except (AttributeError, io.UnsupportedOperation):
    sys.stdout.fileno = lambda: sys.__stdout__.fileno()

# NOTE: we deliberately do NOT touch torch.cuda here. The scorer cell imports
# torch lazily after vLLM loads.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

`TEST_KIND` is the only knob you'll typically flip between runs.


In [ ]:
# =====================================================================
# THE ONE KNOB YOU FLIP BETWEEN RUNS
# =====================================================================

# Run 1: "balanced"  → per-class capped test set; matches the FT balanced pass
# Run 2: "raw"       → population-proportional test set; matches the FT raw pass
TEST_KIND = "raw"
assert TEST_KIND in ("balanced", "raw"), TEST_KIND

# =====================================================================
# Model
# =====================================================================

BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"

# Model identifier used in the output dir + meta.json. Distinct from the
# FT tag so the two runs don't collide in /content/drive/MyDrive/NLP_project/data/.
MODEL_TAG = f"Qwen2.5-14B-Instruct_baseline_test_{TEST_KIND}"

# =====================================================================
# Stage 3 pre-built test split (IDENTICAL to the FT notebook)
# =====================================================================

TEST_SPLIT_DIR = Path(
    "/content/drive/MyDrive/NLP_project/data/"
    "stage3_outputs/stage3_training_datasets"
) / "test" / TEST_KIND

# Fallback: full ACS extract, used only to compute the Hájek-weight column.
# Same path as the FT notebook.
DATA_PATH = Path(
    "/content/drive/MyDrive/NLP_project/data/"
    "demographic_data_clean.parquet"
)

# =====================================================================
# Run knobs
# =====================================================================

DTYPE            = "bfloat16"      # matches the FT run
GPU_MEM_UTIL     = 0.90
TENSOR_PARALLEL  = 1
MAX_MODEL_LEN    = 8192
ENFORCE_EAGER    = True

BATCH_SIZE       = 64
CHECKPOINT_EVERY = 500

# Cap on the number of rows scored per target. Set to None to use the full
# test parquet as written by Stage 3. When set to an integer, each per-target
# test set is downsampled to (at most) this many rows using a stratified
# sample on the target column. Within each class:
#   - TEST_KIND='raw'      → rows drawn with prob ∝ PERWT (population-faithful)
#   - TEST_KIND='balanced' → rows drawn uniformly (don't undo the balancing)
N_SAMPLE_TEST_SET = 5000
SUBSAMPLE_SEED = 5050

# Distinct from the FT seeds (RANDOM_SEED=4343, PERM_SEED_BASE=30_000) for
# book-keeping. The letter permutations themselves come from the Stage 3
# row_seed column on each test parquet — IDENTICAL to what the FT notebook
# sees on the matching row_id — so PERM_SEED_BASE here is only used if a
# row arrives without a row_seed (defensive; shouldn't fire).
PERM_SEED_BASE   = 40_000
RANDOM_SEED      = 4242

PROTOCOL         = "MMLU_letter_permuted"
PRIMING_SUFFIX   = ""

# --- PriDe debiasing ------------------------------------------------------
# Same setting as the FT run, so raw-vs-deb ablation is comparable.
APPLY_PRIDE           = True
PRIDE_ESTIMATION_FRAC = 0.05
PRIDE_N_PERMUTATIONS  = None    # None = use n_Y (full cyclic basis)

# =====================================================================
# Targets — all 7 (matches the FT run)
# =====================================================================

TARGETS_TO_RUN = [
    "SEX",
    "AGE_BINS",
    "EDUCATION_LEVEL",
    "RACE_ETHNICITY",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]

# =====================================================================
# Output dir — TEST_KIND in the tag, distinct from the FT dir
# =====================================================================

MM_DD = datetime.now().strftime("%m_%d")
OUT_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_TAG}_{MM_DD}/"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "!" * 60)
print("CONFIG SUMMARY")
print("!" * 60)
print(f"TEST_KIND       : {TEST_KIND}")
print(f"Base model      : {BASE_MODEL}")
print(f"Adapter         : NONE (this is the baseline)")
print(f"Test split dir  : {TEST_SPLIT_DIR}")
print(f"Output dir      : {OUT_DIR}")
print(f"Targets         : {TARGETS_TO_RUN}")
print(f"N_SAMPLE_TEST_SET: {N_SAMPLE_TEST_SET}")

## 2. Load the Stage 3 test split

Same source files as the FT notebook. Loading them here gives us the
`row_id` / `row_seed` / `true_<TARGET>` columns we need for row-alignment.


In [ ]:
# --- Full ACS extract, used only for pi_c (population class shares) ----
acs_df = pd.read_parquet(DATA_PATH)
assert "PERWT" in acs_df.columns, "ACS extract is missing the PERWT column."
assert (acs_df["PERWT"] > 0).all(), "PERWT contains non-positive values."
print(f"ACS extract: {len(acs_df):,} rows")
print(f"PERWT: min={acs_df['PERWT'].min():.1f}, mean={acs_df['PERWT'].mean():.1f}, "
      f"max={acs_df['PERWT'].max():.1f}")

# --- Per-target test parquets ------------------------------------------
def _allocate_per_class(counts: pd.Series, n_total: int) -> dict:
    """Largest-remainder allocation of n_total rows across classes.

    Allocates proportionally to `counts` (the row-count marginal), guarantees
    >= 1 per present class, sums to exactly n_total. Returns {class: n_take}.
    """
    classes = counts.index.tolist()
    shares = counts.to_numpy() / counts.sum()
    raw_alloc = shares * n_total
    base_alloc = np.floor(raw_alloc).astype(int)
    base_alloc = np.maximum(base_alloc, 1)

    overshoot = int(base_alloc.sum() - n_total)
    if overshoot > 0:
        residuals = raw_alloc - np.floor(raw_alloc)
        order = np.argsort(residuals)  # ascending
        i = 0
        while overshoot > 0 and i < len(order):
            j = order[i]
            if base_alloc[j] > 1:
                base_alloc[j] -= 1
                overshoot -= 1
            else:
                i += 1
        if overshoot > 0:
            order2 = np.argsort(-base_alloc)
            i = 0
            while overshoot > 0:
                j = order2[i % len(order2)]
                if base_alloc[j] > 1:
                    base_alloc[j] -= 1
                    overshoot -= 1
                i += 1
    elif overshoot < 0:
        residuals = raw_alloc - np.floor(raw_alloc)
        order = np.argsort(-residuals)
        i = 0
        while overshoot < 0:
            j = order[i % len(order)]
            base_alloc[j] += 1
            overshoot += 1
            i += 1

    return dict(zip(classes, base_alloc.tolist()))


def _subsample_test_df(df: pd.DataFrame, target_col: str, n_total: int,
                       seed: int, weight_col: str | None = None) -> pd.DataFrame:
    """Stratified downsample of a test parquet to exactly n_total rows.

    Allocates rows to each class proportionally to that class's row count in
    `df` (so the row-count target marginal is preserved). Within each class,
    rows are drawn:
        - uniformly without replacement if `weight_col` is None,
        - with probability proportional to df[weight_col] (without replacement)
          otherwise.

    Guarantees >= 1 row per class that has any rows in `df`.
    """
    if n_total is None or len(df) <= n_total:
        return df

    rng = np.random.default_rng(seed)
    counts = df[target_col].value_counts()
    alloc = _allocate_per_class(counts, n_total)

    parts = []
    for cls, n_take in alloc.items():
        sub = df[df[target_col] == cls]
        n_take = min(int(n_take), len(sub))
        if n_take <= 0:
            continue

        if weight_col is None:
            picked = sub.sample(
                n=n_take,
                random_state=int(rng.integers(0, 2**31 - 1)),
            )
        else:
            w = sub[weight_col].astype(float).to_numpy()
            if (w <= 0).any():
                raise ValueError(
                    f"{weight_col} has non-positive values in class {cls!r}; "
                    f"cannot weight-sample."
                )
            p = w / w.sum()
            idx = rng.choice(
                np.arange(len(sub)),
                size=n_take,
                replace=False,
                p=p,
            )
            picked = sub.iloc[idx]

        parts.append(picked)

    sampled = pd.concat(parts, axis=0).sample(
        frac=1.0, random_state=int(rng.integers(0, 2**31 - 1))
    ).reset_index(drop=True)
    return sampled


# In `raw` mode the Stage 3 split is population-proportional and rows still
# carry their ACS PERWT, so within-class sampling weighted by PERWT keeps
# the joint distribution over context features faithful to the population.
# In `balanced` mode the per-class cap is deliberate — re-weighting by PERWT
# inside a class would partially undo the balancing trick, so we sample
# uniformly there.
SAMPLE_WEIGHT_COL = "PERWT" if TEST_KIND == "raw" else None

test_dfs = {}
for t in TARGETS_TO_RUN:
    path = TEST_SPLIT_DIR / f"{t}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing test parquet for {t}: {path}")
    full = pd.read_parquet(path)

    if N_SAMPLE_TEST_SET is not None and len(full) > N_SAMPLE_TEST_SET:
        wcol = SAMPLE_WEIGHT_COL
        if wcol is not None and wcol not in full.columns:
            raise KeyError(
                f"TEST_KIND={TEST_KIND!r} expects '{wcol}' on the test "
                f"parquet for weighted subsampling, but it is missing in "
                f"{path}."
            )
        sampled = _subsample_test_df(
            full,
            target_col=t,
            n_total=N_SAMPLE_TEST_SET,
            seed=SUBSAMPLE_SEED,
            weight_col=wcol,
        )
        test_dfs[t] = sampled
        wtag = f"weighted by {wcol}" if wcol else "uniform"
        print(f"  {t:<20} loaded {len(full):,} rows  → sampled {len(sampled):,} "
              f"({wtag})  ({path})")
    else:
        test_dfs[t] = full
        print(f"  {t:<20} loaded {len(full):,} rows  ({path})")

# Quick row-alignment sanity check: every target test parquet should have
# a row_seed column inherited from Stage 3. If any are missing, the
# downstream paired tests will lose their row alignment.
for t, df in test_dfs.items():
    if "row_seed" not in df.columns:
        print(f"  WARNING: {t} test parquet has no row_seed column — "
              f"row alignment with FT will be broken for paired tests.")


## 3. Feature dictionaries

Identical to the FT notebook. This is critical: the prompts must be
byte-for-byte the same as what the FT model saw, otherwise the comparison
isn't measuring the FT effect but rather (FT effect + prompt drift).


In [ ]:
FEATURE_LABELS = {
    "SEX":              "Sex",
    "AGE_BINS":         "Age group",
    "RACE_ETHNICITY":   "Race/ethnicity",
    "EDUCATION_LEVEL":  "Highest level of education completed",
    "INCTOT_BINS":      "Total annual personal income",
    "HEALTH_INSURANCE": "Health insurance coverage",
    "ENGLISH_STATUS":   "English speaking ability",
    "NCHILD":           "Number of own children living in household",
    "UHRSWORK":         "Usual hours worked per week",
    "STATE_NAME":       "State of residence",
    "OWNERSHP_STATUS":  "Home ownership status",
    "MARITAL_STATUS":   "Marital status",
    "EMPLOYMENT_STATUS":"Employment status",
    "WORKER_CLASS":     "Class of worker",
    "TRANSPORT_METHOD": "Means of transportation to work",
}

In [ ]:
TARGET_OPTION_ORDER = {
    "SEX": ["Male", "Female"],
    "AGE_BINS": ["18–24", "25–34", "35–44", "45–54", "55–64", "65+"],
    "RACE_ETHNICITY": [
        "White", "Hispanic", "Black", "Other",
        "Chinese", "Indigenous", "Japanese",
    ],
    "EDUCATION_LEVEL": [],
    "INCTOT_BINS": [],
    "HEALTH_INSURANCE": [],
    "ENGLISH_STATUS": [],
}

LABEL_TO_LONG = {
    "White":      "White",
    "Hispanic":   "Hispanic",
    "Black":      "Black or African American",
    "Indigenous": "American Indian or Alaska Native",
    "Chinese":    "Chinese",
    "Japanese":   "Japanese",
    "Other":      "Other Asian or Pacific Islander",
    "18–24":      "18 to 24 years old",
    "25–34":      "25 to 34 years old",
    "35–44":      "35 to 44 years old",
    "45–54":      "45 to 54 years old",
    "55–64":      "55 to 64 years old",
    "65+":        "65 years or older",
    "Male":       "Male",
    "Female":     "Female",
}

In [ ]:
def _resolve_class_labels(df: pd.DataFrame, target_col: str,
                          acs_df: pd.DataFrame = None) -> list:
    """Canonical class order for target_col.

    Identical to the FT notebook — critical for row alignment, since the
    letter permutation depends on class order.
    """
    present = set(df[target_col].dropna().unique())

    if acs_df is not None and "PERWT" in acs_df.columns:
        perwt_by_cls = (
            acs_df.dropna(subset=[target_col])
                  .groupby(target_col, observed=True)["PERWT"].sum()
        )
        nonzero = set(perwt_by_cls[perwt_by_cls > 0].index)
        present = present & nonzero

    canonical = TARGET_OPTION_ORDER.get(target_col, []) or []

    if canonical:
        missing = [c for c in canonical if c not in present]
        if missing:
            print(
                f"WARNING: TARGET_OPTION_ORDER[{target_col!r}] lists classes "
                f"not present in test set: {missing}. Dropping them."
            )
            canonical = [c for c in canonical if c in present]
        extras = sorted(present - set(canonical))
        if extras:
            print(
                f"WARNING: {target_col} has classes not in canonical order, "
                f"appending alphabetically: {extras}"
            )
        return list(canonical) + extras

    return sorted(present)


def _display_label(class_label: str) -> str:
    return LABEL_TO_LONG.get(class_label, str(class_label))

## 4. Prepare the eval set from the test split

Identical to the FT notebook. The key invariant: this notebook honors the
Stage 3 `row_seed` column when present, so the per-row `letter_map_json`
is **byte-for-byte identical** to the FT notebook's letter map on the same
`row_id`. That's what makes paired tests meaningful — the baseline and FT
see the same letter-to-class assignment on every row.


In [ ]:
def _acs_class_shares(acs_df: pd.DataFrame, target_col: str,
                      class_labels: list) -> pd.Series:
    valid = acs_df.dropna(subset=[target_col])
    weighted = valid.groupby(target_col, observed=True)["PERWT"].sum()
    pi = weighted / valid["PERWT"].sum()
    pi = pi.reindex(class_labels)
    if pi.isna().any():
        missing = pi[pi.isna()].index.tolist()
        raise ValueError(
            f"No ACS rows found for classes {missing} of {target_col}; "
            f"check class label canonicalization."
        )
    return pi


def make_letter_map(class_labels: list, row_seed: int) -> dict:
    rng = np.random.default_rng(int(row_seed))
    n = len(class_labels)
    letters = list(string.ascii_uppercase[:n])
    shuffled = list(class_labels)
    rng.shuffle(shuffled)
    return dict(zip(letters, shuffled))


def prepare_eval_set(
    test_df: pd.DataFrame,
    acs_df: pd.DataFrame,
    target_col: str,
    class_labels: list,
    perm_seed_base: int = PERM_SEED_BASE,
) -> pd.DataFrame:
    df = test_df.copy()

    if df[target_col].isna().any():
        n_dropped = int(df[target_col].isna().sum())
        print(f"  WARNING: dropping {n_dropped} rows with NaN {target_col}")
        df = df.dropna(subset=[target_col]).reset_index(drop=True)

    if "row_seed" not in df.columns:
        print(f"  WARNING: {target_col} test parquet has no row_seed column; "
              f"synthesizing from perm_seed_base — row alignment with FT BROKEN.")
        df["row_seed"] = perm_seed_base + np.arange(len(df), dtype=np.int64)

    pi = _acs_class_shares(acs_df, target_col, class_labels)
    print(f"\n[{target_col}] ACS class shares pi_c:")
    for c in class_labels:
        print(f"  {c!s:<32} pi_c = {pi[c]:.4f}")

    N = len(df)
    n_c_realized = df[target_col].value_counts()
    pi_per_row  = df[target_col].map(pi).astype(float)
    n_c_per_row = df[target_col].map(n_c_realized).astype(float)
    df["hajek_weight"] = pi_per_row / n_c_per_row * N

    df["letter_map_json"] = [
        json.dumps(make_letter_map(class_labels, int(row_seed)))
        for row_seed in df["row_seed"].to_numpy()
    ]

    print(f"  N = {len(df):,}")
    weighted_share = (
        df.groupby(target_col, observed=True)["hajek_weight"].sum()
        / df["hajek_weight"].sum()
    )
    drift = (weighted_share.reindex(class_labels) - pi).abs().max()
    print(f"  Max |weighted share - pi_c| = {drift:.2e}  (should be ~0)")

    return df

## 5. Prompt construction

Verbatim from the FT notebook (which in turn matches the Stage 3 training
prompts). Any drift here breaks the comparison.


In [ ]:





SYSTEM_PROMPT = """You are completing one missing demographic attribute for a U.S. survey respondent.

You will be given a profile with one missing attribute and a list of lettered answer options. Predict the most likely value of that missing attribute using the other provided attributes.

Rules:
- Use only the provided profile.
- Choose exactly one option from the list.
- Reply with only the letter of the selected option — no parentheses, no explanation, no extra text."""


def _excluded_columns(target_col: str) -> set:
    return {target_col}


def build_profile_lines(row: pd.Series, target_col: str) -> list:
    excluded = _excluded_columns(target_col)
    lines = []
    for col, label in FEATURE_LABELS.items():
        if col in excluded or col not in row.index:
            continue
        value = row[col]
        if pd.isna(value):
            continue
        display_value = LABEL_TO_LONG.get(value, value)
        lines.append(f"- {label}: {display_value}")
    return lines


def build_user_prompt(row: pd.Series, target_col: str, lettered_options: list) -> str:
    profile_text = "\n".join(build_profile_lines(row, target_col))
    answers_text = "\n".join(f"({letter}) {label}" for letter, label in lettered_options)
    target_label = FEATURE_LABELS[target_col]

    return (
        f"Profile:\n\n{profile_text}\n\n"
        f"Missing attribute: {target_label}\n\n"
        f"Answer options:\n{answers_text}\n\n"
        f"Reply with only the letter of the correct option."
    )

In [ ]:
# Visual sanity check: render two prompts for the first target.
_first_target = TARGETS_TO_RUN[0]
_first_classes = _resolve_class_labels(test_dfs[_first_target], _first_target, acs_df)
print(f"Resolved classes for {_first_target}: {_first_classes}\n")

_preview_df = test_dfs[_first_target]
for _i in range(min(2, len(_preview_df))):
    _row = _preview_df.iloc[_i]
    _seed = int(_row["row_seed"]) if "row_seed" in _row.index else PERM_SEED_BASE + _i
    _lm = make_letter_map(_first_classes, row_seed=_seed)
    _lettered = [(L, _display_label(c)) for L, c in _lm.items()]
    print(build_user_prompt(_row, _first_target, _lettered))
    print("-" * 60)

## 6. LLM scorer (vLLM)

Same scorer class as the FT notebook. `model_name` is the HuggingFace ID
for `Qwen/Qwen2.5-14B-Instruct`; vLLM downloads and caches it on first run.


In [ ]:
def cleanup_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


class LLMScorer:
    """vLLM-backed MMLU-style letter scorer."""

    def __init__(
        self,
        model_name: str,
        dtype: str = DTYPE,
        gpu_memory_utilization: float = GPU_MEM_UTIL,
        tensor_parallel_size: int = TENSOR_PARALLEL,
        max_model_len: int = MAX_MODEL_LEN,
        enforce_eager: bool = ENFORCE_EAGER,
    ):
        from vllm import LLM
        from transformers import AutoTokenizer

        cleanup_memory()

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.chat_template is None:
            jinja_path = Path(model_name) / "chat_template.jinja"
            if jinja_path.exists():
                self.tokenizer.chat_template = jinja_path.read_text()
                print(f"Loaded chat_template from sidecar: {jinja_path}")
            else:
                raise RuntimeError(
                    f"Tokenizer at {model_name} has no chat_template and "
                    f"no chat_template.jinja sidecar."
                )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.llm = LLM(
            model=model_name,
            dtype=dtype,
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=gpu_memory_utilization,
            max_model_len=max_model_len,
            enforce_eager=enforce_eager,
        )

    def _build_prefix(self, system_prompt: str, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    @staticmethod
    def _assert_single_token(tokenizer, letter: str) -> int:
        ids = tokenizer(letter, add_special_tokens=False).input_ids
        if len(ids) != 1:
            raise AssertionError(
                f"Letter {letter!r} tokenizes to {len(ids)} tokens; "
                "MMLU mode requires single-token letters."
            )
        return ids[0]

    def score_letters_batched(
        self,
        prompts: list,
        letters_per_prompt: list,
        priming_suffix: str = "",
    ) -> list:
        from vllm import SamplingParams

        full_prompts = []
        index = []
        for prof_idx, ((sys_p, usr_p), letters) in enumerate(zip(prompts, letters_per_prompt)):
            prefix = self._build_prefix(sys_p, usr_p) + priming_suffix
            for letter in letters:
                full_prompts.append(prefix + letter)
                index.append((prof_idx, letter))

        sampling_params = SamplingParams(
            temperature=0.0,
            max_tokens=1,
            prompt_logprobs=20,
        )
        outputs = self.llm.generate(full_prompts, sampling_params, use_tqdm=False)

        results = [{} for _ in prompts]
        for out_idx, (prof_idx, letter) in enumerate(index):
            out = outputs[out_idx]
            letter_id = self._assert_single_token(self.tokenizer, letter)

            last_pos = len(out.prompt_token_ids) - 1
            token_logprobs = out.prompt_logprobs[last_pos]
            if letter_id in token_logprobs:
                lp = token_logprobs[letter_id].logprob
            else:
                lp = -100.0
            results[prof_idx][letter] = lp

        return results

## 7. Per-target scoring driver

Identical to the FT notebook. Output schema is byte-for-byte the same, so
the eval notebooks (`0`–`4`) work on the baseline `SCORED_DIR` without any
changes.


In [ ]:
def _logsumexp(x: np.ndarray) -> float:
    m = x.max()
    return float(m + np.log(np.exp(x - m).sum()))


def score_target(
    scorer: "LLMScorer",
    test_df: pd.DataFrame,
    acs_df: pd.DataFrame,
    target_col: str,
    out_dir: Path = OUT_DIR,
    model_tag: str = "",
    model_path: str = "",
    batch_size: int = BATCH_SIZE,
    checkpoint_every: int = CHECKPOINT_EVERY,
    priming_suffix: str = PRIMING_SUFFIX,
    seed: int = RANDOM_SEED,
    perm_seed_base: int = PERM_SEED_BASE,
    apply_pride: bool = APPLY_PRIDE,
    pride_estimation_frac: float = PRIDE_ESTIMATION_FRAC,
    pride_n_permutations: int = PRIDE_N_PERMUTATIONS,
) -> pd.DataFrame:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 78)
    print(f"# Target: {target_col}  (baseline, PriDe={'on' if apply_pride else 'off'})")
    print("#" * 78)

    class_labels = _resolve_class_labels(test_df, target_col, acs_df)
    n_classes = len(class_labels)
    letters = list(string.ascii_uppercase[:n_classes])
    print(f"Classes ({n_classes}): {class_labels}")

    letter_token_ids = {
        L: scorer._assert_single_token(scorer.tokenizer, L) for L in letters
    }

    eval_df = prepare_eval_set(
        test_df,
        acs_df=acs_df,
        target_col=target_col,
        class_labels=class_labels,
        perm_seed_base=perm_seed_base,
    )
    eval_df = eval_df.reset_index(drop=True)
    eval_set_path = out_dir / f"phase1_eval_set_{target_col}.parquet"
    eval_df.to_parquet(eval_set_path)
    print(f"Saved eval set → {eval_set_path}")

    # --- PriDe estimation/inference split --------------------------------
    K = int(pride_n_permutations) if pride_n_permutations is not None else n_classes
    rng_split = np.random.default_rng(seed + 7)

    n_eval = len(eval_df)
    n_est_target = max(n_classes, int(round(pride_estimation_frac * n_eval)))
    n_est_per_cls = max(1, n_est_target // n_classes)

    est_idx_set = set()
    for cls in class_labels:
        cls_pool = eval_df.index[eval_df[target_col] == cls].to_numpy()
        if len(cls_pool) < n_est_per_cls:
            chosen = cls_pool
        else:
            chosen = rng_split.choice(cls_pool, size=n_est_per_cls, replace=False)
        est_idx_set.update(int(i) for i in chosen)
    est_indices = np.array(sorted(est_idx_set), dtype=np.int64)
    inf_indices = np.array(
        [i for i in range(n_eval) if i not in est_idx_set], dtype=np.int64
    )
    print(f"PriDe split: |estimation|={len(est_indices)}  "
          f"|inference|={len(inf_indices)}  K={K}")

    # --- Inference prompts ----------------------------------------------
    inf_prompts = []
    inf_letter_maps = []
    for row_idx in inf_indices:
        row = eval_df.iloc[row_idx]
        lm = json.loads(row["letter_map_json"])
        lettered_display = [(L, _display_label(c)) for L, c in lm.items()]
        up = build_user_prompt(row, target_col, lettered_display)
        inf_prompts.append((SYSTEM_PROMPT, up))
        inf_letter_maps.append(lm)

    # --- Estimation prompts ---------------------------------------------
    est_prompts = []
    est_letter_maps_per_perm = []
    for row_idx in est_indices:
        row = eval_df.iloc[row_idx]
        base_lm = json.loads(row["letter_map_json"])
        base_classes = [base_lm[L] for L in letters]
        per_row_maps = []
        for k in range(K):
            shifted = base_classes[k:] + base_classes[:k]
            lm_k = dict(zip(letters, shifted))
            lettered_display = [(L, _display_label(c)) for L, c in lm_k.items()]
            up = build_user_prompt(row, target_col, lettered_display)
            est_prompts.append((SYSTEM_PROMPT, up))
            per_row_maps.append(lm_k)
        est_letter_maps_per_perm.append(per_row_maps)

    checkpoint_path = out_dir / f"phase1_scored_{target_col}_partial.parquet"
    final_path      = out_dir / f"phase1_scored_{target_col}.parquet"

    # --- PASS 1a: estimation set -----------------------------------------
    n_est = len(est_indices)
    est_logits = np.zeros((n_est, K, n_classes), dtype=np.float64)
    pbar = tqdm(total=len(est_prompts),
                desc=f"Scoring {target_col} (1a/2: PriDe estimation)")
    for batch_start in range(0, len(est_prompts), batch_size):
        batch_end = min(batch_start + batch_size, len(est_prompts))
        batch_logprobs = scorer.score_letters_batched(
            est_prompts[batch_start:batch_end],
            [letters] * (batch_end - batch_start),
            priming_suffix=priming_suffix,
        )
        for offset, raw_letter_lp in enumerate(batch_logprobs):
            flat_idx = batch_start + offset
            i_local = flat_idx // K
            k = flat_idx %  K
            for j, L in enumerate(letters):
                est_logits[i_local, k, j] = raw_letter_lp[L]
        pbar.update(batch_end - batch_start)
    pbar.close()

    mean_letter_logits = est_logits.mean(axis=(0, 1))
    log_prior = mean_letter_logits - _logsumexp(mean_letter_logits)
    letter_prior = {L: float(log_prior[i]) for i, L in enumerate(letters)}

    print(f"\nPriDe per-target letter prior (held-out, K={K}) for {target_col}:")
    for L, lp in letter_prior.items():
        print(f"  L={L}: log P_prior = {lp:+.4f}  (P_prior = {np.exp(lp):.4f}, "
              f"uniform = {1.0 / n_classes:.4f})")

    per_row_letter_std = est_logits.std(axis=1).mean(axis=0)
    prior_spread = log_prior.max() - log_prior.min()
    print(f"  per-row letter-logit std: "
          f"{np.array2string(per_row_letter_std, precision=3)}")
    print(f"  prior spread max-min log P_prior: {prior_spread:+.3f}")

    # --- PASS 1b: inference set -----------------------------------------
    inf_letter_logits = np.zeros((len(inf_indices), n_classes), dtype=np.float64)
    pbar = tqdm(total=len(inf_prompts),
                desc=f"Scoring {target_col} (1b/2: inference)")
    inf_done = 0
    for batch_idx, batch_start in enumerate(range(0, len(inf_prompts), batch_size)):
        batch_end = min(batch_start + batch_size, len(inf_prompts))
        batch_logprobs = scorer.score_letters_batched(
            inf_prompts[batch_start:batch_end],
            [letters] * (batch_end - batch_start),
            priming_suffix=priming_suffix,
        )
        for offset, raw_letter_lp in enumerate(batch_logprobs):
            i_local = batch_start + offset
            for j, L in enumerate(letters):
                inf_letter_logits[i_local, j] = raw_letter_lp[L]
        inf_done += (batch_end - batch_start)
        pbar.update(batch_end - batch_start)

        if (batch_idx + 1) % checkpoint_every == 0:
            partial_df = pd.DataFrame(
                inf_letter_logits[:inf_done],
                columns=[f"logprob_letter_{L}" for L in letters],
            )
            partial_df.to_parquet(checkpoint_path)
            pbar.set_postfix(checkpoint=f"row {inf_done}")
    pbar.close()

    # --- PASS 2: debias and renormalize ---------------------------------
    floor = -99.0
    n_clipped_inf = int((inf_letter_logits < floor).sum())
    n_total_inf   = int(inf_letter_logits.size)
    n_clipped_est = int((est_logits < floor).sum())
    n_total_est   = int(est_logits.size)
    print(f"\nClipped letter-logprobs (== -100 floor):")
    print(f"  estimation: {n_clipped_est}/{n_total_est} "
          f"({100 * n_clipped_est / max(n_total_est, 1):.2f}%)")
    print(f"  inference : {n_clipped_inf}/{n_total_inf} "
          f"({100 * n_clipped_inf / max(n_total_inf, 1):.2f}%)")

    def _make_record(lm, raw_letter_lp_arr, role, true_cls):
        class_to_letter = {c: L for L, c in lm.items()}
        assert len(class_to_letter) == n_classes, "letter map is not bijective"
        cls_letters = [class_to_letter[c] for c in class_labels]
        letter_to_idx = {L: i for i, L in enumerate(letters)}
        ll_class = np.array(
            [raw_letter_lp_arr[letter_to_idx[L]] for L in cls_letters],
            dtype=np.float64,
        )

        probs_raw = np.exp(ll_class - ll_class.max())
        probs_raw /= probs_raw.sum()

        prior_at_letter = np.array(
            [letter_prior[L] for L in cls_letters], dtype=np.float64
        )
        ll_deb = ll_class - prior_at_letter
        probs_deb = np.exp(ll_deb - ll_deb.max())
        probs_deb /= probs_deb.sum()

        probs_canonical = probs_deb if apply_pride else probs_raw
        pred_cls = class_labels[int(np.argmax(probs_canonical))]

        rec = {"pride_role": role}
        for j, L in enumerate(letters):
            rec[f"logprob_letter_{L}"] = float(raw_letter_lp_arr[j])
        for j, c in enumerate(class_labels):
            rec[f"logprob_raw_{c}"] = float(ll_class[j])
            rec[f"prob_raw_{c}"]    = float(probs_raw[j])
            rec[f"prob_deb_{c}"]    = float(probs_deb[j])
            rec[f"prob_{c}"]        = float(probs_canonical[j])
        rec[f"true_{target_col}"] = true_cls
        rec[f"pred_{target_col}"] = pred_cls
        rec["p_true"] = float(probs_canonical[class_labels.index(true_cls)])
        return rec

    records_by_row = [None] * n_eval

    for i_local, row_idx in enumerate(est_indices):
        lm = est_letter_maps_per_perm[i_local][0]
        raw_letter_lp_arr = est_logits[i_local, 0, :]
        true_cls = eval_df.iloc[row_idx][target_col]
        records_by_row[row_idx] = _make_record(
            lm, raw_letter_lp_arr, role="estimation", true_cls=true_cls
        )

    for i_local, row_idx in enumerate(inf_indices):
        lm = inf_letter_maps[i_local]
        raw_letter_lp_arr = inf_letter_logits[i_local, :]
        true_cls = eval_df.iloc[row_idx][target_col]
        records_by_row[row_idx] = _make_record(
            lm, raw_letter_lp_arr, role="inference", true_cls=true_cls
        )

    scored_df = pd.concat(
        [eval_df, pd.DataFrame(records_by_row)], axis=1
    )
    scored_df.to_parquet(final_path)

    meta = {
        "target":               target_col,
        "model_name":           model_tag,
        "model_path":           str(model_path),
        "is_baseline":          True,
        "dtype":                DTYPE,
        "protocol":             PROTOCOL,
        "priming_suffix":       priming_suffix,
        "class_labels":         class_labels,
        "letters_used":         letters,
        "label_to_long":        {c: _display_label(c) for c in class_labels},
        "eval_set_source":      "stage3_test_split",
        "test_kind":            TEST_KIND,
        "test_split_path":      str(TEST_SPLIT_DIR / f"{target_col}.parquet"),
        "n_eval_rows":          int(len(scored_df)),
        "batch_size":           batch_size,
        "random_seed":          seed,
        "perm_seed_base":       perm_seed_base,
        "system_prompt":        SYSTEM_PROMPT,
        "context_features":     [c for c in FEATURE_LABELS if c != target_col],
        "apply_pride":          bool(apply_pride),
        "canonical_prob":       "deb" if apply_pride else "raw",
        "pride_n_estimation":   int(n_est),
        "pride_n_inference":    int(len(inf_indices)),
        "pride_n_permutations": int(K),
        "pride_estimation_frac_target": float(pride_estimation_frac),
        "letter_log_prior":     letter_prior,
        "letter_prior_uniform": 1.0 / n_classes,
        "clipped_letter_logprobs": {
            "estimation_count": n_clipped_est,
            "estimation_total": n_total_est,
            "inference_count":  n_clipped_inf,
            "inference_total":  n_total_inf,
        },
        "timestamp":            pd.Timestamp.now().isoformat(),
    }
    meta_path = final_path.with_suffix(".meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2, default=str)

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print(f"Saved scored data → {final_path}")
    print(f"Saved metadata    → {meta_path}")

    inf_only = scored_df[scored_df["pride_role"] == "inference"]
    acc_raw = (
        inf_only[f"true_{target_col}"]
        == inf_only[[f"prob_raw_{c}" for c in class_labels]]
            .idxmax(axis=1).str.replace("prob_raw_", "", regex=False)
    ).mean()
    acc_deb = (
        inf_only[f"true_{target_col}"]
        == inf_only[[f"prob_deb_{c}" for c in class_labels]]
            .idxmax(axis=1).str.replace("prob_deb_", "", regex=False)
    ).mean()
    p_true_raw = inf_only.apply(
        lambda r: r[f"prob_raw_{r[f'true_{target_col}']}"], axis=1
    ).mean()
    p_true_canon = inf_only["p_true"].mean()
    print(f"\nSanity diagnostics (inference rows only):")
    print(f"  Argmax accuracy raw : {acc_raw:.3f}")
    print(f"  Argmax accuracy deb : {acc_deb:.3f}")
    print(f"  Mean p_true raw     : {p_true_raw:.3f}")
    print(f"  Mean p_true (canon) : {p_true_canon:.3f}   "
          f"(canonical = {'debiased' if apply_pride else 'raw'})")
    print(f"  Random baseline     : {1 / n_classes:.3f}")

    return scored_df

## 8. Load base model and run all 7 targets

vLLM downloads `Qwen/Qwen2.5-14B-Instruct` from the HuggingFace Hub on first
run. The model is ~28 GB in bf16; expect a few minutes for the initial
download to `~/.cache/huggingface/hub/`. On Colab this lives on the ephemeral
disk by default — the cache will be repopulated each new runtime.


In [ ]:
scorer = LLMScorer(
    model_name=BASE_MODEL,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEM_UTIL,
    tensor_parallel_size=TENSOR_PARALLEL,
    max_model_len=MAX_MODEL_LEN,
    enforce_eager=ENFORCE_EAGER,
)

In [ ]:
scored_by_target = {}

for target in TARGETS_TO_RUN:
    final_path = OUT_DIR / f"phase1_scored_{target}.parquet"
    if final_path.exists():
        print(f"\n[{target}] Already scored at {final_path} — skipping.")
        scored_by_target[target] = pd.read_parquet(final_path)
        continue

    scored_by_target[target] = score_target(
        scorer=scorer,
        test_df=test_dfs[target],
        acs_df=acs_df,
        target_col=target,
        out_dir=OUT_DIR,
        model_tag=MODEL_TAG,
        model_path=BASE_MODEL,
        seed=RANDOM_SEED,
        perm_seed_base=PERM_SEED_BASE,
    )

print("\n✓ All targets evaluated.")
print(f"Outputs under: {OUT_DIR}")

## 9. Row-alignment sanity check (optional)

If you've already run the FT notebook with the same `TEST_KIND`, this cell
verifies that the baseline and FT output parquets agree on `row_id`,
`row_seed`, `true_<TARGET>`, and `letter_map_json` for every row — i.e.
the two runs are genuinely row-aligned and paired tests are well-defined.

Point `FT_DIR` at the matching FT output directory. If you haven't run the
FT notebook yet, just skip this cell.


In [ ]:
# Point at the matching FT output (same TEST_KIND).
# Leave as None to skip the check.
FT_DIR = None
# Example:
# FT_DIR = Path(
#     f"/content/drive/MyDrive/NLP_project/data/"
#     f"stage_1_output_Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_{TEST_KIND}_05_11"
# )

if FT_DIR is None:
    print("FT_DIR not set — skipping row-alignment check.")
else:
    assert Path(FT_DIR).is_dir(), f"Missing FT dir: {FT_DIR}"
    print(f"Checking row alignment between\n  baseline: {OUT_DIR}\n  FT:       {FT_DIR}\n")
    for target in TARGETS_TO_RUN:
        bp = OUT_DIR / f"phase1_scored_{target}.parquet"
        fp = Path(FT_DIR) / f"phase1_scored_{target}.parquet"
        if not (bp.exists() and fp.exists()):
            print(f"  [{target}] missing parquet on one side — skipping")
            continue

        b = pd.read_parquet(bp)
        f = pd.read_parquet(fp)

        cols = ["row_id", "row_seed", f"true_{target}", "letter_map_json"]
        cols = [c for c in cols if c in b.columns and c in f.columns]

        if len(b) != len(f):
            print(f"  [{target}] LENGTH MISMATCH: baseline={len(b)}, FT={len(f)} — "
                  f"paired tests broken for this target")
            continue

        # Sort both by row_id for a stable comparison, then compare the shared cols.
        if "row_id" in cols:
            b = b.sort_values("row_id").reset_index(drop=True)
            f = f.sort_values("row_id").reset_index(drop=True)

        all_match = True
        for c in cols:
            if not (b[c].astype(str) == f[c].astype(str)).all():
                n_diff = int((b[c].astype(str) != f[c].astype(str)).sum())
                print(f"  [{target}] {c}: {n_diff}/{len(b)} rows differ")
                all_match = False

        if all_match:
            print(f"  [{target}] ✓ {len(b):,} rows fully aligned on {cols}")

## 10. Release Colab runtime (optional)

In [ ]:
# from google.colab import runtime
# runtime.unassign()

---

**Next steps:**

1. After this `TEST_KIND="balanced"` run, flip `TEST_KIND = "raw"` in cell §1,
   restart the kernel, and re-run.
2. Once both passes are done for both the baseline (this notebook) and the FT
   (`stage_1_inference_on_FT_natural_canonical.ipynb`), you'll have four
   `SCORED_DIR`s laid out as:

       stage_1_output_Qwen2.5-14B-Instruct_baseline_test_balanced_…/
       stage_1_output_Qwen2.5-14B-Instruct_baseline_test_raw_…/
       stage_1_output_…_FT_natural_canonical_step07_bf16_test_balanced_…/
       stage_1_output_…_FT_natural_canonical_step07_bf16_test_raw_…/

3. The eval notebooks (`0`–`4`) work unchanged on any of these — point
   `SCORED_DIR` at whichever run you want to evaluate.
4. For **paired tests** (McNemar, paired bootstrap on Δp_true, per-row
   improvement decomposition), you'll need a new notebook that loads two
   `SCORED_DIR`s simultaneously, joins on `row_id`, and computes per-row
   deltas. The row-alignment check in §9 confirms that this join is
   well-defined.


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
from google.colab import runtime
runtime.unassign()